# 03b · Inteligencia: IA en texto, pronóstico y red de abasto
> **Workshop Tec de Monterrey · De los datos a los agentes** — Abarrotera del Centro S.A. (empresa ficticia, datos sintéticos).

**Pregunta de negocio:** ya sabemos *qué* pasó. ¿Qué dicen los clientes en una línea, cuánto vamos a perder si no hacemos nada y desde dónde mandamos producto?

| Parte | Capacidad Snowflake | Resultado |
|---|---|---|
| 1 | `AI_CLASSIFY`, `AI_SENTIMENT`, `AI_AGG` | Tema y sentimiento de cada comentario + resumen ejecutivo gobernado |
| 2 | `SNOWFLAKE.ML.FORECAST` | Venta esperada de Bebidas vs real y escenario de noviembre sin acción |
| 3 | `ST_DISTANCE` (geoespacial) | CEDIS más cercano con excedente para surtir al CEDIS León |

Todo queda en tablas que el agente consulta por la Semantic View (04).

In [ ]:
%%sql -r ctx_base_esquema
-- Contexto de ejecución (primera celda). Lo caro (IA en vivo, entrenamiento y pronóstico) usa IF NOT EXISTS: re-ejecutar
-- no repite ese costo; las tablas derivadas sí se recalculan (CREATE OR REPLACE) para reflejar los datos actuales.
USE ROLE TEC_ADMIN;
USE WAREHOUSE TEC_WH;
USE DATABASE TEC_DEMO;
USE SCHEMA TEC_DEMO.ANALITICA;

-- Fechas de la historia en UN solo lugar (la app usa los mismos valores en lib/constants.ts)
CREATE OR REPLACE TABLE TEC_DEMO.ANALITICA.PARAMETROS (parametro STRING, valor DATE, descripcion STRING) AS
SELECT * FROM VALUES
  ('corte_entrenamiento', '2026-09-30'::DATE, 'Último día con el que se entrena el pronóstico (antes del quiebre)'),
  ('mes_analisis',        '2026-10-01'::DATE, 'Mes de la caída de Bebidas en el Bajío'),
  ('mes_pronostico',      '2026-11-01'::DATE, 'Mes del escenario "si no actuamos"'),
  ('inicio_pronostico',   '2026-05-01'::DATE, 'Primer mes de la comparación real vs esperado'),
  ('fecha_inventario',    '2026-10-31'::DATE, 'Foto de inventario para calcular excedentes de la red de abasto');
SET (corte_entrenamiento, mes_analisis, mes_pronostico, inicio_pronostico, fecha_inventario) = (
  SELECT MAX(IFF(parametro = 'corte_entrenamiento', valor, NULL)), MAX(IFF(parametro = 'mes_analisis', valor, NULL)),
         MAX(IFF(parametro = 'mes_pronostico', valor, NULL)), MAX(IFF(parametro = 'inicio_pronostico', valor, NULL)),
         MAX(IFF(parametro = 'fecha_inventario', valor, NULL))
  FROM TEC_DEMO.ANALITICA.PARAMETROS);

## Parte 1 · IA sobre 60 mil comentarios
Los tenderos escriben textos repetidos: clasificamos cada texto **distinto** una sola vez (centavos de crédito) y lo unimos a los 60 mil comentarios. Así el agente puede contar quejas por tema.

In [ ]:
-- 1) EN VIVO: la IA clasifica los 16 textos distintos (centavos de crédito)
CREATE TABLE IF NOT EXISTS TEMAS_COMENTARIO_EN_VIVO AS
SELECT comentario,
  AI_CLASSIFY(comentario, ['Desabasto de producto', 'Entrega y logística', 'Precio y competencia', 'Surtido y variedad',
                           'Atención y servicio', 'Crédito y pagos', 'Producto y promoción'],
              {'task_description': 'Clasifica el comentario de un tendero mexicano sobre su distribuidor de abarrotes'}):labels[0]::STRING AS tema,
  AI_SENTIMENT(comentario):categories[0]:sentiment::STRING AS sentimiento
FROM (SELECT DISTINCT comentario FROM TEC_DEMO.RAW.FEEDBACK_CLIENTES);

-- 2) OFICIAL: la misma clasificación congelada en el kit (un LLM puede variar entre cuentas; así todos ven las mismas cifras)
--    comentario_id = HASH(comentario): la misma llave que VOZ_CLIENTE, para unir por número y no por texto.
CREATE TABLE IF NOT EXISTS TEMAS_COMENTARIO (comentario STRING, tema STRING, sentimiento STRING, comentario_id NUMBER);
ALTER TABLE TEMAS_COMENTARIO ADD COLUMN IF NOT EXISTS comentario_id NUMBER;  -- instalaciones anteriores
CALL TEC_DEMO.RAW.CARGAR_CSV('TEMAS_COMENTARIO');  -- carpeta y huella: TEC_DEMO.RAW.HUELLA_CARGA
UPDATE TEMAS_COMENTARIO SET comentario_id = HASH(comentario) WHERE comentario_id IS NULL;

-- 3) ¿Coincide tu IA con la oficial?
SELECT o.tema AS tema_oficial, v.tema AS tema_en_vivo, o.sentimiento, IFF(o.tema = v.tema, '✓', '≠') AS coincide, o.comentario
FROM TEMAS_COMENTARIO o LEFT JOIN TEMAS_COMENTARIO_EN_VIVO v USING (comentario)
ORDER BY coincide DESC, tema_oficial;

In [ ]:
%%sql -r quejas_por_tema
SELECT t.tema, COUNT(*) AS quejas, ROUND(AVG(v.calificacion), 2) AS calificacion_promedio
FROM VOZ_CLIENTE v JOIN TEMAS_COMENTARIO t ON t.comentario_id = v.comentario_id  -- llave numérica, no el texto
WHERE v.region = 'Bajío' AND v.mes = '2026-10-01' AND v.calificacion <= 2
GROUP BY t.tema ORDER BY quejas DESC;

In [ ]:
%%sql -r resumen_quejas
-- AI_AGG lee TODAS las filas del grupo (no una muestra) y escribe un resumen. Respeta la Row Access Policy de VOZ_CLIENTE
SELECT AI_AGG(comentario, 'Eres analista comercial. Resume en 3 viñetas cortas, en español, qué problema reportan estos tenderos y qué piden. Cierra con una recomendación de una línea.') AS resumen
FROM VOZ_CLIENTE
WHERE region = 'Bajío' AND mes = '2026-10-01' AND calificacion <= 2;

In [ ]:
%%sql -r resumen_norte
-- La IA solo resume lo que el rol puede ver: el gerente del Norte no obtiene nada del Bajío
EXECUTE USING POLICY_CONTEXT(CURRENT_ROLE => 'GERENTE_NORTE')
  AS SELECT COUNT(*) AS comentarios_visibles,
            IFF(COUNT(*) = 0, 'Sin comentarios visibles para este rol', 'Hay comentarios visibles') AS resultado
     FROM TEC_DEMO.ANALITICA.VOZ_CLIENTE
     WHERE region = 'Bajío' AND mes = '2026-10-01' AND calificacion <= 2;

## Parte 2 · ¿Cuánto vamos a perder si no hacemos nada?
Entrenamos `SNOWFLAKE.ML.FORECAST` con la venta diaria de Bebidas **hasta el 30-sep-2026** (antes del problema), una serie por región. El modelo dice cuánto *debimos* vender en octubre y cuánto esperar en noviembre. La diferencia es el costo del quiebre.

In [ ]:
%%sql -r bebidas_diarias
-- Serie diaria de Bebidas por región. CREATE OR REPLACE: si recargas los datos, se recalcula (la celda gobierno_nuevas
-- le vuelve a poner la Row Access Policy y el permiso). El modelo de abajo NO se reentrena (IF NOT EXISTS).
CREATE OR REPLACE TABLE BEBIDAS_DIARIAS AS
SELECT region, fecha::TIMESTAMP_NTZ AS ts, SUM(venta_neta) AS venta
FROM VENTAS_ENRIQUECIDAS WHERE categoria = 'Bebidas'
GROUP BY region, fecha;
-- Entrenar (~1 min la primera vez). Solo datos hasta el 30-sep: el modelo no "sabe" del quiebre
CREATE SNOWFLAKE.ML.FORECAST IF NOT EXISTS MODELO_BEBIDAS(
  INPUT_DATA => TABLE(SELECT region, ts, venta FROM BEBIDAS_DIARIAS WHERE ts <= (SELECT valor FROM TEC_DEMO.ANALITICA.PARAMETROS WHERE parametro = 'corte_entrenamiento')),
  SERIES_COLNAME => 'region', TIMESTAMP_COLNAME => 'ts', TARGET_COLNAME => 'venta');

In [ ]:
-- 1) EN VIVO: tu modelo pronostica oct–nov 2026. Se lee como función de tabla (sin depender de LAST_QUERY_ID)
--    e IF NOT EXISTS: re-ejecutar no vuelve a pagar el pronóstico.
CREATE TABLE IF NOT EXISTS PRONOSTICO_DIARIO_EN_VIVO AS
SELECT series::STRING AS region, ts, forecast AS venta_esperada, lower_bound, upper_bound
FROM TABLE(MODELO_BEBIDAS!FORECAST(FORECASTING_PERIODS => 61, CONFIG_OBJECT => {'prediction_interval': 0.9}));

-- 2) OFICIAL: el pronóstico congelado en el kit (el entrenamiento puede variar ligeramente entre cuentas)
CREATE TABLE IF NOT EXISTS PRONOSTICO_DIARIO (region STRING, ts TIMESTAMP_NTZ, venta_esperada FLOAT, lower_bound FLOAT, upper_bound FLOAT);
CALL TEC_DEMO.RAW.CARGAR_CSV('PRONOSTICO_DIARIO');  -- carpeta y huella: TEC_DEMO.RAW.HUELLA_CARGA

-- 3) Tu modelo vs el oficial: venta esperada de octubre por región (M MXN)
SELECT o.region, ROUND(SUM(o.venta_esperada) / 1e6, 2) AS oficial_mdp, ROUND(SUM(v.venta_esperada) / 1e6, 2) AS en_vivo_mdp,
       ROUND(100 * (SUM(v.venta_esperada) / SUM(o.venta_esperada) - 1), 1) AS diferencia_pct
FROM PRONOSTICO_DIARIO o LEFT JOIN PRONOSTICO_DIARIO_EN_VIVO v ON v.region = o.region AND v.ts = o.ts
WHERE o.ts < '2026-11-01' GROUP BY o.region ORDER BY o.region;

In [ ]:
%%sql -r pronostico_mensual
-- Real vs esperado por mes y región. Noviembre: esperado (sin quiebre) y escenario "sin acción" (mismo % de pérdida que octubre)
-- CREATE OR REPLACE: se recalcula si cambian BEBIDAS_DIARIAS o el pronóstico oficial.
-- LEFT JOIN: una región sin dato de octubre se conserva (con venta_sin_accion nula) en vez de desaparecer.
CREATE OR REPLACE TABLE PRONOSTICO_MENSUAL AS
WITH r AS (SELECT region, DATE_TRUNC('month', ts)::DATE AS mes, SUM(venta) AS venta_real FROM BEBIDAS_DIARIAS
           WHERE ts >= $inicio_pronostico GROUP BY 1, 2),
     e AS (SELECT region, DATE_TRUNC('month', ts)::DATE AS mes, SUM(venta_esperada) AS venta_esperada,
                  SUM(lower_bound) AS esperada_baja, SUM(upper_bound) AS esperada_alta FROM PRONOSTICO_DIARIO GROUP BY 1, 2),
     m AS (SELECT COALESCE(r.region, e.region) AS region, COALESCE(r.mes, e.mes) AS mes, 'Bebidas' AS categoria,
                  r.venta_real, e.venta_esperada, e.esperada_baja, e.esperada_alta
           FROM r FULL OUTER JOIN e ON r.region = e.region AND r.mes = e.mes),
     f AS (SELECT region, venta_real / NULLIF(venta_esperada, 0) AS factor_oct FROM m WHERE mes = $mes_analisis)
SELECT m.*, IFF(m.mes = $mes_pronostico, ROUND(m.venta_esperada * LEAST(f.factor_oct, 1), 0), NULL) AS venta_sin_accion
FROM m LEFT JOIN f USING (region);
SELECT * FROM PRONOSTICO_MENSUAL WHERE region = 'Bajío' ORDER BY mes;

In [ ]:
%%sql -r gobierno_nuevas
-- Misma regla por región en TODAS las tablas derivadas que traen región (DROP ALL + ADD: idempotente)
ALTER TABLE PRONOSTICO_MENSUAL DROP ALL ROW ACCESS POLICIES;
ALTER TABLE PRONOSTICO_MENSUAL ADD ROW ACCESS POLICY RAP_REGION ON (region);
ALTER TABLE BEBIDAS_DIARIAS DROP ALL ROW ACCESS POLICIES;
ALTER TABLE BEBIDAS_DIARIAS ADD ROW ACCESS POLICY RAP_REGION ON (region);
ALTER TABLE PRONOSTICO_DIARIO DROP ALL ROW ACCESS POLICIES;
ALTER TABLE PRONOSTICO_DIARIO ADD ROW ACCESS POLICY RAP_REGION ON (region);
-- TEMAS_COMENTARIO no lleva política: es el catálogo de los 16 textos con su tema (no tiene región ni datos de tiendas);
-- la región vive en VOZ_CLIENTE, que ya tiene RAP_REGION.
GRANT SELECT ON TABLE PRONOSTICO_MENSUAL TO ROLE LECTOR_COMERCIAL;
GRANT SELECT ON TABLE BEBIDAS_DIARIAS TO ROLE LECTOR_COMERCIAL;
GRANT SELECT ON TABLE PRONOSTICO_DIARIO TO ROLE LECTOR_COMERCIAL;
GRANT SELECT ON TABLE TEMAS_COMENTARIO TO ROLE LECTOR_COMERCIAL;

In [ ]:
import pandas as pd
import plotly.graph_objects as go

pm = pronostico_mensual.to_pandas() if hasattr(pronostico_mensual, 'to_pandas') else pronostico_mensual.copy()
pm.columns = [c.lower() for c in pm.columns]
b = pm[pm['region'] == 'Bajío'].sort_values('mes')
fig = go.Figure()
fig.add_trace(go.Scatter(x=b['mes'], y=b['esperada_alta'] / 1e6, line={'width': 0}, showlegend=False, hoverinfo='skip'))
fig.add_trace(go.Scatter(x=b['mes'], y=b['esperada_baja'] / 1e6, fill='tonexty', line={'width': 0}, name='Rango esperado (90%)', fillcolor='rgba(41,128,185,0.15)'))
fig.add_trace(go.Scatter(x=b['mes'], y=b['venta_esperada'] / 1e6, name='Esperado (modelo)', line={'dash': 'dash', 'color': '#2980b9'}))
fig.add_trace(go.Scatter(x=b['mes'], y=b['venta_real'] / 1e6, name='Real', mode='lines+markers', line={'color': '#2c3e50', 'width': 3}))
fig.add_trace(go.Scatter(x=b['mes'], y=b['venta_sin_accion'] / 1e6, name='Noviembre sin acción', mode='markers', marker={'size': 14, 'color': '#c0392b', 'symbol': 'x'}))
fig.update_layout(title='Bebidas Bajío: venta real vs pronóstico (millones MXN)', height=480, yaxis_title='MXN millones')
fig.show()
oct_ = b[b['mes'].astype(str).str.startswith('2026-10')].iloc[0]
nov_ = b[b['mes'].astype(str).str.startswith('2026-11')].iloc[0]
print(f"Octubre: esperado ${oct_.venta_esperada/1e6:.1f} M, real ${oct_.venta_real/1e6:.1f} M → brecha ${(oct_.venta_esperada-oct_.venta_real)/1e6:.1f} M")
print(f"Noviembre: esperado ${nov_.venta_esperada/1e6:.1f} M; sin acción ${nov_.venta_sin_accion/1e6:.1f} M → en riesgo ${(nov_.venta_esperada-nov_.venta_sin_accion)/1e6:.1f} M")

## Parte 3 · ¿Desde qué CEDIS surtimos a León?
El procedimiento de abasto dice: *"reasignar inventario en 72 horas desde el CEDIS más cercano"*. Con `ST_DISTANCE` (km en línea recta × 1.25 por carretera, camión a 65 km/h) y el excedente de Bebidas de cada CEDIS al cierre de octubre, la decisión sale de los datos.

In [ ]:
%%sql -r red_abasto
-- CREATE OR REPLACE: se recalcula si recargas inventario o CEDIS (abajo se vuelven a poner política y permiso)
CREATE OR REPLACE TABLE RED_ABASTO AS
WITH inv AS (
  SELECT cedis_id, SUM(GREATEST(unidades_disponibles - demanda_diaria * 7, 0)) AS excedente_bebidas_unidades,
         ROUND(AVG(IFF(en_quiebre, 1, 0)) * 100, 1) AS quiebre_pct
  FROM TEC_DEMO.RAW.INVENTARIO_DIARIO WHERE categoria = 'Bebidas' AND fecha = $fecha_inventario GROUP BY cedis_id)
SELECT o.nombre AS cedis_origen, o.region AS region_origen, d.nombre AS cedis_destino, d.region AS region_destino,
       ROUND(ST_DISTANCE(o.geo, d.geo) / 1000) AS distancia_km,
       ROUND(ST_DISTANCE(o.geo, d.geo) / 1000 * 1.25 / 65, 1) AS horas_camion,
       io.excedente_bebidas_unidades AS excedente_origen_unidades, io.quiebre_pct AS quiebre_origen_pct,
       o.latitud AS lat_origen, o.longitud AS lon_origen, d.latitud AS lat_destino, d.longitud AS lon_destino
FROM TEC_DEMO.RAW.CEDIS o JOIN TEC_DEMO.RAW.CEDIS d ON o.cedis_id <> d.cedis_id
JOIN inv io ON io.cedis_id = o.cedis_id;
-- Cada gerente ve las rutas que SURTEN a su región (region_destino): el Bajío ve qué CEDIS le pueden mandar a León.
ALTER TABLE RED_ABASTO DROP ALL ROW ACCESS POLICIES;
ALTER TABLE RED_ABASTO ADD ROW ACCESS POLICY RAP_REGION ON (region_destino);
GRANT SELECT ON TABLE RED_ABASTO TO ROLE LECTOR_COMERCIAL;
-- Opciones para surtir al CEDIS León (el mapa de abajo usa este resultado)
SELECT * FROM RED_ABASTO WHERE cedis_destino = 'CEDIS León' ORDER BY distancia_km;

In [ ]:
# Mapa sin mapa base de internet: go.Scatter con ejes longitud/latitud (se dibuja igual que la gráfica de arriba)
# México = contorno simplificado embebido (lon, lat; ~80 vértices, ilustrativo). Mar = fondo azul. Puntos grises = 20,000 tiendas
import plotly.graph_objects as go
from snowflake.snowpark.context import get_active_session

MEXICO = [(-117.12,32.53),(-114.72,32.72),(-114.81,32.49),(-111.07,31.33),(-108.21,31.33),(-108.21,31.78),(-106.51,31.75),
  (-104.92,30.60),(-104.40,29.57),(-103.10,28.97),(-102.40,29.77),(-101.40,29.77),(-100.50,28.30),(-99.50,27.50),(-99.10,26.40),
  (-97.50,25.90),(-97.40,25.00),(-97.70,24.00),(-97.80,22.30),(-97.20,20.60),(-96.10,19.20),(-95.20,18.70),(-94.40,18.15),
  (-92.60,18.60),(-91.40,18.60),(-90.70,19.30),(-90.40,20.90),(-90.00,21.20),(-88.40,21.60),(-87.00,21.50),(-86.80,20.60),
  (-87.50,19.50),(-87.80,18.30),(-88.30,18.50),(-89.15,17.95),(-90.98,17.82),(-91.00,17.25),(-91.40,16.07),(-92.20,15.25),
  (-92.24,14.53),(-93.40,15.60),(-94.60,16.20),(-95.20,16.17),(-96.50,15.65),(-98.00,16.10),(-99.90,16.85),(-101.60,17.60),
  (-102.40,17.95),(-103.50,18.30),(-104.30,19.10),(-105.30,19.90),(-105.40,20.55),(-105.20,21.50),(-105.70,22.60),(-106.40,23.20),
  (-107.40,24.40),(-108.60,25.40),(-109.40,26.10),(-109.90,27.00),(-110.60,27.90),(-111.30,28.40),(-112.20,29.00),(-112.80,30.10),
  (-113.10,31.20),(-114.80,31.80),(-114.70,30.90),(-114.00,29.60),(-113.30,28.80),(-112.50,27.80),(-111.60,26.60),(-110.70,24.90),
  (-110.30,24.20),(-109.40,23.40),(-109.90,22.90),(-110.90,24.00),(-112.10,24.80),(-112.10,25.90),(-113.20,26.70),(-114.10,27.60),
  (-114.20,28.00),(-115.80,29.80),(-116.30,30.50),(-116.60,31.80),(-117.12,32.53)]

session = get_active_session()
fondo = session.sql("SELECT latitud::FLOAT AS lat, longitud::FLOAT AS lon FROM TEC_DEMO.RAW.TIENDAS").to_pandas()
fondo.columns = [c.lower() for c in fondo.columns]

leon = red_abasto.to_pandas() if hasattr(red_abasto, 'to_pandas') else red_abasto.copy()
leon.columns = [c.lower() for c in leon.columns]
for c in ['lat_origen', 'lon_origen', 'lat_destino', 'lon_destino', 'distancia_km', 'horas_camion', 'excedente_origen_unidades']:
    leon[c] = leon[c].astype(float)
leon = leon.sort_values('distancia_km').reset_index(drop=True)
mejor = leon.iloc[0]

fig = go.Figure()
# Territorio nacional
fig.add_trace(go.Scatter(x=[p[0] for p in MEXICO], y=[p[1] for p in MEXICO], fill='toself', mode='lines',
                         fillcolor='#f6f1e7', line={'color': '#a6a197', 'width': 1}, name='México', hoverinfo='skip', showlegend=False))
fig.add_trace(go.Scatter(x=fondo['lon'], y=fondo['lat'], mode='markers', name='Tiendas (20,000)',
                         marker={'size': 3, 'color': '#c9c3b6'}, hoverinfo='skip'))
for r in leon.itertuples():
    es_mejor = r.cedis_origen == mejor.cedis_origen
    fig.add_trace(go.Scatter(x=[r.lon_origen, r.lon_destino], y=[r.lat_origen, r.lat_destino], mode='lines',
                             line={'width': 6 if es_mejor else 2, 'color': '#27ae60' if es_mejor else '#7f8c8d', 'dash': 'solid' if es_mejor else 'dot'},
                             name=f"{r.cedis_origen}: {r.distancia_km:.0f} km · {r.horas_camion:.1f} h", hoverinfo='name'))
# Etiquetas separadas para que León y Tultitlán no se encimen
posiciones = {'CEDIS Tultitlán': 'bottom right', 'CEDIS Monterrey': 'top center', 'CEDIS Veracruz': 'bottom right', 'CEDIS Mérida': 'top center'}
fig.add_trace(go.Scatter(x=leon['lon_origen'], y=leon['lat_origen'], mode='markers+text', text=leon['cedis_origen'],
                         textposition=[posiciones.get(c, 'top center') for c in leon['cedis_origen']],
                         marker={'size': 14, 'color': '#2c3e50'}, showlegend=False,
                         customdata=leon[['distancia_km', 'horas_camion', 'excedente_origen_unidades']],
                         hovertemplate='%{text}<br>%{customdata[0]:.0f} km · %{customdata[1]:.1f} h<br>Excedente: %{customdata[2]:,.0f} u<extra></extra>'))
fig.add_trace(go.Scatter(x=[mejor.lon_destino], y=[mejor.lat_destino], mode='markers+text', text=['CEDIS León<br>34.6% quiebre'],
                         textposition='top left', marker={'size': 20, 'color': '#c0392b', 'symbol': 'star'}, showlegend=False, hoverinfo='text'))
fig.add_annotation(x=-93.5, y=24.5, text='Golfo de México', showarrow=False, font={'color': '#5d7fa3', 'size': 12})
fig.add_annotation(x=-110.5, y=18.0, text='Océano Pacífico', showarrow=False, font={'color': '#5d7fa3', 'size': 12})
fig.update_xaxes(range=[-118, -86], showgrid=False, zeroline=False, visible=False)
fig.update_yaxes(range=[14, 33], showgrid=False, zeroline=False, visible=False, scaleanchor='x', scaleratio=1)
fig.update_layout(title=f"Reasignar desde {mejor.cedis_origen}: {mejor.distancia_km:.0f} km, ~{mejor.horas_camion:.0f} h de camión",
                  height=560, plot_bgcolor='#dcebf7', legend={'x': 0.01, 'y': 0.02, 'bgcolor': 'rgba(255,255,255,0.85)'},
                  margin={'l': 10, 'r': 10, 't': 50, 'b': 10})
fig.show()

**Lectura:**
- **IA en texto:** de las 328 quejas del Bajío en octubre, **274 son por desabasto** y 54 por precio/competencia. `AI_AGG` lo resume en un párrafo, y solo con lo que el rol puede ver.
- **Pronóstico:** en octubre debimos vender **$24.2 M** en Bebidas del Bajío y vendimos $15.8 M: **$8.4 M perdidos**. Si no actuamos, noviembre deja **$9.1 M más** en la mesa.
- **Red:** el CEDIS más cercano con excedente es **Tultitlán: 310 km, ~6 h de camión, 88 mil unidades disponibles**. Cumple el plazo de 72 h del procedimiento con mucha holgura.

En el notebook 04 estas tres tablas entran a la Semantic View: el agente ya puede responder *"¿cuánto vamos a perder en noviembre?"* y *"¿desde qué CEDIS reasigno?"*.